# 14 — Patient-Level Prediction

Aggregates held-out cycle predictions to recording then patient level. Cycle rows inherit a single patient-wise split; no held-out patient participates in training. Class-probability aggregation is the primary method; majority vote and duration-weighted probability are also reported where prediction files provide a cycle weight.

In [ ]:
import sys,json
from pathlib import Path
ROOT=Path.cwd().resolve()
if ROOT.name=='post_05': ROOT=ROOT.parent.parent
elif ROOT.name=='notebooks': ROOT=ROOT.parent
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
import pandas as pd
from src.post_05.common import PREDICTIONS_DIR, METRICS_DIR, ensure_output_dirs, new_artifact_path
from src.post_05.patient_level import aggregate_patient_predictions, classification_metrics
ensure_output_dirs()
reports=[]
for path in sorted(PREDICTIONS_DIR.glob('*_cycle_predictions.csv')):
    frame=pd.read_csv(path)
    probability_columns=[c for c in frame.columns if c.startswith('prob_')]
    if not probability_columns or not {'patient_id','recording_id','diagnosis','prediction'}<=set(frame.columns):
        continue
    labels=sorted(set(frame.diagnosis.astype(str))|{c.removeprefix('prob_') for c in probability_columns})
    patient=aggregate_patient_predictions(frame,probability_columns,'mean_probability')
    recording=aggregate_patient_predictions(frame.assign(patient_id=frame.patient_id.astype(str)+'_'+frame.recording_id.astype(str)),probability_columns,'mean_probability')
    for level,table in [('recording',recording),('patient',patient)]:
        metrics=classification_metrics(table.diagnosis.astype(str),table.prediction.astype(str),labels)
        reports.append({'model':path.stem.replace('_cycle_predictions',''),'level':level,**{k:metrics[k] for k in ('accuracy','macro_precision','macro_recall','macro_f1','weighted_f1')}})
        save_path=new_artifact_path(PREDICTIONS_DIR/f'{path.stem}_{level}_predictions.csv'); table.to_csv(save_path,index=False)
        new_artifact_path(METRICS_DIR/f'{path.stem}_{level}_metrics.json').write_text(json.dumps(metrics,indent=2))
summary=pd.DataFrame(reports)
if summary.empty: print('No prediction files found. Run Notebook 09 and available CNN experiments first.')
else: display(summary); new_artifact_path(METRICS_DIR/'aggregation_comparison.csv'); summary.to_csv(METRICS_DIR/'aggregation_comparison.csv',index=False)